# 03 Train Models

Train LightGBM, LSTM, and GRU from the same processed feature matrix. LSTM and GRU checkpoints are saved after every epoch and resume from `*_latest.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

%cd $REPO_DIR
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

Run this in the browser console when a long training run starts:

```javascript
function keepAlive() {
  document.querySelector("colab-toolbar-button#connect").click();
}
setInterval(keepAlive, 60000);
```

In [ ]:
import json

import torch

from config import FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig, SplitConfig
from features.pipeline import FeatureEngineeringPipeline
from features.sequences import SequenceBuilder
from models.lgbm import LightGBMModel
from models.splits import chronological_split
from models.torch_models import GRUClassifier, LSTMClassifier
from models.torch_training import result_to_dict, train_torch_classifier, write_torch_training_metadata

LOOKBACK = 60
EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
NUM_WORKERS = 2

paths = PathConfig(root=BASE, raw_data_dir=DATA_DIR / 'raw', processed_data_dir=DATA_DIR / 'processed', artifact_dir=ARTIFACT_DIR, model_artifact_dir=MODEL_DIR, report_dir=REPORT_DIR)
if (DATA_DIR / 'processed' / 'BANKNIFTY_5m_features.parquet').exists():
    market = MarketConfig(symbol='BANKNIFTY', ticker='BANKNIFTY', series='IDX', interval='5m', intraday_source='drive-cache-local-csv', daily_source='intraday-resample')
elif (DATA_DIR / 'processed' / 'NIFTY_5m_features.parquet').exists():
    market = MarketConfig(symbol='NIFTY', ticker='NIFTY', series='IDX', interval='5m', intraday_source='drive-cache-local-csv', daily_source='intraday-resample')
else:
    market = MarketConfig(symbol='RELIANCE', ticker='RELIANCE.NS', series='EQ', interval='5m', intraday_source='drive-cache-or-jugaad-openchart')
pipeline = FeatureEngineeringPipeline(paths=paths, market=market, features=FeatureConfig(), labels=LabelConfig(), normalizer=NormalizerConfig())
dataset = pipeline.load()
splits = chronological_split(dataset.frame, SplitConfig())

print(f'Rows train={len(splits.train):,} val={len(splits.validation):,} test={len(splits.test):,}')
print(f'Features={len(dataset.feature_columns):,}')
print('Device:', torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

In [ ]:
lgbm_trainer = LightGBMModel(paths=paths, market=market)
lgbm_result = lgbm_trainer.train(dataset.frame, feature_columns=dataset.feature_columns, split_config=SplitConfig())
print(lgbm_result.model_path)
print(lgbm_result.validation_metrics)
print(lgbm_result.test_metrics)

In [ ]:
sequence_builder = SequenceBuilder(lookback=LOOKBACK)
train_arrays = sequence_builder.build(splits.train, feature_columns=dataset.feature_columns)
validation_arrays = sequence_builder.build(splits.validation, feature_columns=dataset.feature_columns)
test_arrays = sequence_builder.build(splits.test, feature_columns=dataset.feature_columns)

input_size = len(dataset.feature_columns)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(train_arrays.x.shape, validation_arrays.x.shape, test_arrays.x.shape)

In [ ]:
lstm = LSTMClassifier(input_size=input_size, hidden_size_1=128, hidden_size_2=64, dropout=0.3)
lstm_result = train_torch_classifier(
    model=lstm,
    model_name='lstm',
    train_arrays=train_arrays,
    validation_arrays=validation_arrays,
    model_dir=MODEL_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_workers=NUM_WORKERS,
    device=device,
)
print(lstm_result)

In [ ]:
if device.type == 'cuda':
    torch.cuda.empty_cache()

gru = GRUClassifier(input_size=input_size, hidden_size=128, dropout=0.3)
gru_result = train_torch_classifier(
    model=gru,
    model_name='gru',
    train_arrays=train_arrays,
    validation_arrays=validation_arrays,
    model_dir=MODEL_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_workers=NUM_WORKERS,
    device=device,
)
print(gru_result)

In [ ]:
training_metadata = {
    'ticker': market.ticker,
    'interval': market.interval,
    'lookback': LOOKBACK,
    'input_size': input_size,
    'feature_config': str(pipeline.feature_config_path()),
    'feature_columns': dataset.feature_columns,
    'lgbm_model': str(lgbm_result.model_path),
    'lstm': result_to_dict(lstm_result),
    'gru': result_to_dict(gru_result),
}
write_torch_training_metadata(MODEL_DIR / 'torch_training_metadata.json', training_metadata)
print(MODEL_DIR / 'torch_training_metadata.json')